# Backfill 1850 values
Small adjustment to finished QDM dataset for use in protocol testing.  Step2 dropped Jan-Nov 1850 timesteps during de-meaning, as annual means assigned to "year end" timestep before reconstruction.

For now: Duplicate 1851 and use it to fill in 1850 for every time step.

1 Oct 2025 | EHU

In [1]:
import xarray as xr
from datetime import datetime
import cftime as cftime

In [20]:
DirIn = f'/Users/eultee/Desktop/'

input_file = f'{DirIn}/TF_aQDM-ISMIP_Grid-CESM2-WACCM-1850_2014-PFromStep1-20250926.nc'
ds_in = xr.open_dataset(input_file, decode_times='timeDim')
ds_in

<xarray.Dataset> Size: 38GB
Dimensions:  (time: 1969, y: 2881, x: 1681)
Coordinates:
  * time     (time) object 16kB 1850-12-31 00:00:00 ... 2014-12-31 00:00:00
  * y        (y) float32 12kB -3.45e+06 -3.449e+06 ... -5.71e+05 -5.7e+05
  * x        (x) float32 7kB -7.2e+05 -7.19e+05 -7.18e+05 ... 9.59e+05 9.6e+05
Data variables:
    TF       (time, y, x) float32 38GB ...
Attributes:
    title:          Ocean thermal forcing for CESM2-WACCM
    summary:        TF computed following Verjans bias correction and Slater ...
    institution:    NASA Goddard Space Flight Center
    creation_date:  2025-09-26 10:56:51

In [21]:

ds_1850_cp = ds_in.sel(time=slice(cftime.DatetimeNoLeap(1851, 1, 1), cftime.DatetimeNoLeap(1851, 11, 30)))
ds_1850_cp

<xarray.Dataset> Size: 213MB
Dimensions:  (time: 11, y: 2881, x: 1681)
Coordinates:
  * time     (time) object 88B 1851-01-31 00:00:00 ... 1851-11-30 00:00:00
  * y        (y) float32 12kB -3.45e+06 -3.449e+06 ... -5.71e+05 -5.7e+05
  * x        (x) float32 7kB -7.2e+05 -7.19e+05 -7.18e+05 ... 9.59e+05 9.6e+05
Data variables:
    TF       (time, y, x) float32 213MB ...
Attributes:
    title:          Ocean thermal forcing for CESM2-WACCM
    summary:        TF computed following Verjans bias correction and Slater ...
    institution:    NASA Goddard Space Flight Center
    creation_date:  2025-09-26 10:56:51

In [22]:
new_time_axis = xr.date_range(start='1850-01-31', end='1850-11-30', freq='ME', use_cftime=True)
ds_1850_cp = ds_1850_cp.assign_coords(new_time = ('time', new_time_axis))
ds_1850_cp = ds_1850_cp.drop_indexes('time').drop_vars('time')
ds_1850_cp.set_xindex('new_time')
ds_1850_cp

<xarray.Dataset> Size: 213MB
Dimensions:   (time: 11, y: 2881, x: 1681)
Coordinates:
  * y         (y) float32 12kB -3.45e+06 -3.449e+06 ... -5.71e+05 -5.7e+05
  * x         (x) float32 7kB -7.2e+05 -7.19e+05 -7.18e+05 ... 9.59e+05 9.6e+05
    new_time  (time) object 88B 1850-01-31 00:00:00 ... 1850-11-30 00:00:00
Dimensions without coordinates: time
Data variables:
    TF        (time, y, x) float32 213MB ...
Attributes:
    title:          Ocean thermal forcing for CESM2-WACCM
    summary:        TF computed following Verjans bias correction and Slater ...
    institution:    NASA Goddard Space Flight Center
    creation_date:  2025-09-26 10:56:51

In [23]:
ds_1850_cp = ds_1850_cp.rename({'new_time': 'time'})
ds_1850_cp = ds_1850_cp.set_xindex('time')
ds_1850_cp

/var/folders/lm/d_79zh011jddr2c41jsw0dvr0000gq/T/ipykernel_66461/2427722543.py:1: UserWarning: rename 'new_time' to 'time' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  ds_1850_cp = ds_1850_cp.rename({'new_time': 'time'})


<xarray.Dataset> Size: 213MB
Dimensions:  (time: 11, y: 2881, x: 1681)
Coordinates:
  * y        (y) float32 12kB -3.45e+06 -3.449e+06 ... -5.71e+05 -5.7e+05
  * x        (x) float32 7kB -7.2e+05 -7.19e+05 -7.18e+05 ... 9.59e+05 9.6e+05
  * time     (time) object 88B 1850-01-31 00:00:00 ... 1850-11-30 00:00:00
Data variables:
    TF       (time, y, x) float32 213MB ...
Attributes:
    title:          Ocean thermal forcing for CESM2-WACCM
    summary:        TF computed following Verjans bias correction and Slater ...
    institution:    NASA Goddard Space Flight Center
    creation_date:  2025-09-26 10:56:51

In [24]:
ds_1850_cp = ds_1850_cp.convert_calendar('noleap')

In [25]:
ds_out = xr.concat((ds_1850_cp, ds_in), dim='time')
ds_out

<xarray.Dataset> Size: 38GB
Dimensions:  (time: 1980, y: 2881, x: 1681)
Coordinates:
  * y        (y) float32 12kB -3.45e+06 -3.449e+06 ... -5.71e+05 -5.7e+05
  * x        (x) float32 7kB -7.2e+05 -7.19e+05 -7.18e+05 ... 9.59e+05 9.6e+05
  * time     (time) object 16kB 1850-01-31 00:00:00 ... 2014-12-31 00:00:00
Data variables:
    TF       (time, y, x) float32 38GB 6.386 6.386 6.386 6.386 ... 0.0 0.0 0.0
Attributes:
    title:          Ocean thermal forcing for CESM2-WACCM
    summary:        TF computed following Verjans bias correction and Slater ...
    institution:    NASA Goddard Space Flight Center
    creation_date:  2025-09-26 10:56:51

In [26]:
## SINGLE OUTPUT FILE
## write to output file
from datetime import datetime

now = datetime.now()

## Make filename tags showing time for the output 
## with cftime axes:
FirstYear = ds_out.time[0].values.item().year
LastYear = ds_out.time[-1].values.item().year

p_tag = 'PFromStep1'
SelModel='CESM2-WACCM'

## file name
out_fn = DirIn + 'TF_aQDM-ISMIP_Grid-{}-{}_{}-{}-{}.nc'.format(SelModel, 
                                                    FirstYear, LastYear, 
                                                    p_tag, 
                                                    now.strftime('%Y%m%d'))

# ds_temp = tf_out.to_dataset(name='TF')
ds_out = ds_out.assign_attrs(title='Ocean thermal forcing for {}'.format(SelModel),
                             summary='TF computed following Verjans bias correction and Slater inland mapping,' + 
                             'in a bounding box around Greenland, for ISMIP7 Greenland forcing.' +
                                # ' Mean correction based on EN4, 1985-2014.' +
                                 'Additive QDM correction based on EN4, 1985-2014.' +
                                ' Process code: github.com/ehultee/gris-iceocean-process',
                             institution='NASA Goddard Space Flight Center',
                             creation_date=now.strftime('%Y-%m-%d %H:%M:%S'))

## write it!
ds_out.to_netcdf(path=out_fn,
                encoding={'TF': {'zlib': True, 'complevel':9}}) ## set compression level

In [27]:
## check it

fp = out_fn
ds_check = xr.open_dataset(fp)
ds_check

<xarray.Dataset> Size: 38GB
Dimensions:  (time: 1980, y: 2881, x: 1681)
Coordinates:
  * y        (y) float32 12kB -3.45e+06 -3.449e+06 ... -5.71e+05 -5.7e+05
  * x        (x) float32 7kB -7.2e+05 -7.19e+05 -7.18e+05 ... 9.59e+05 9.6e+05
  * time     (time) object 16kB 1850-01-31 00:00:00 ... 2014-12-31 00:00:00
Data variables:
    TF       (time, y, x) float32 38GB ...
Attributes:
    title:          Ocean thermal forcing for CESM2-WACCM
    summary:        TF computed following Verjans bias correction and Slater ...
    institution:    NASA Goddard Space Flight Center
    creation_date:  2025-10-01 11:29:05

In [ ]:
## what does it look like?
## only for visualizations
import cartopy.crs as ccrs ## map projections
import matplotlib.pyplot as plt
import numpy as np

date_toplot = cftime.DatetimeNoLeap(1850, 1, 31)
x_toplot, y_toplot = np.meshgrid(ds_check.x, ds_check.y)

TF_toplot_written = ds_check.TF.sel(time=date_toplot, method='nearest')
TF_toplot_orig = ds_in.TF.sel(time=cftime.DatetimeNoLeap(1851, 1, 31), method='nearest')

# obs_x, obs_y = np.meshgrid(tobs_ds_full.lon, tobs_ds_full.lat)

### Limits of Greenland domain ###
limN           = 86.0 ## degrees N latitude
limS           = 57.0 ## degrees N latitude
limE           = 4.0 ## degrees E latitude
limW           = 274.0 ## degrees E latitude

GrIS_polar_stereo = ccrs.Stereographic(
            central_latitude=90.0,
            central_longitude=-45.0,
            false_easting=0.0,
            false_northing=0.0,
            true_scale_latitude=70.0,
            globe=ccrs.Globe('WGS84')
        )

fig, axs = plt.subplots(1,2, layout='constrained',
                        subplot_kw={'projection': GrIS_polar_stereo, 'extent':  [-65, -20, limS, limN]}
                       )

sc_written = axs[0].scatter(y=y_toplot, x=x_toplot, 
                        c=TF_toplot_written, transform=ccrs.epsg(3413),
                        cmap='viridis', vmin=0., vmax=15.);
# axs[0].plot(TF_toplot_written, 
#             y=TF_toplot_written.y, x=TF_toplot_written.x,
#             transform=ccrs.epsg(3413))
axs[0].set(aspect=1, title='Newly written')
axs[0].coastlines()
cbar1 = plt.colorbar(sc_written, orientation='vertical', label='TF [°C]', shrink=0.4)

sc_diff = axs[1].scatter(y=y_toplot, x=x_toplot, 
                        c=TF_toplot_written-TF_toplot_orig, 
                         transform=ccrs.epsg(3413),
                        cmap='RdBu', vmin=-5., vmax=5.);
axs[1].set(aspect=1, title='Diff from original')
axs[1].coastlines()
# cbar2 = plt.colorbar(sc_obs, orientation='vertical', label='TF [°C]', shrink=0.4)

# # sc_obsdiff = axs[2].scatter(y=TF_toplot_qdm.lat, x=TF_toplot_qdm.lon, 
# #                         c=TF_toplot_qdm-TF_toplot_obs, transform=ccrs.PlateCarree(),
# #                         cmap='RdBu', vmin=-5., vmax=5.);
# # axs[2].set(aspect=1, title='diff QDM - EN4')
# # axs[2].coastlines()
# cbar2 = plt.colorbar(sc_diff, orientation='vertical', label='TF [°C]', shrink=0.4)

fig.suptitle('{}'.format(date_toplot))